# Phase 1: Model Prototyping

This notebook is for interactively developing and testing the core components of our transformer model. Following our agreed workflow, we will:

1.  **Implement** model `nn.Module` classes in the `src/` directory (e.g., `src/model.py`).
2.  **Import** those classes here in the notebook.
3.  **Test** them with sample data to verify their correctness (e.g., checking tensor shapes, data flow, and outputs).

Let's start by loading our dataset and creating a simple character-level tokenizer.

## 1. Data Loading and Tokenization

In [ ]:
# Read the dataset
with open('../data/tinyshakespeare/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("Length of dataset in characters: ", len(text))

In [ ]:
print(text[:100])

Now, we'll create a simple character-level tokenizer.

In [ ]:
# Get all the unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f"Vocabulary size: {vocab_size}")

In [ ]:
# Create a mapping from characters to integers (stoi) and integers to characters (itos)
stoi: dict[str, int] = {ch: i for i, ch in enumerate(chars)}
itos: dict[int, str] = {i: ch for i, ch in enumerate(chars)}

# Encoder:
def encode(s: str) -> list[int]:
	return [stoi[c] for c in s]

# Decoder:
def decode(l: list[int]) -> str:
	return ''.join([itos[i] for i in l])

# Test the tokenizer
test_string = "hello world"
encoded_string = encode(test_string)
decoded_string = decode(encoded_string)

print(f"Original: '{test_string}'")
print(f"Encoded: {encoded_string}")
print(f"Decoded: '{decoded_string}'")

With the tokenizer ready, we can now convert our entire dataset into a `torch.Tensor`.

In [ ]:
import torch

# Encode the entire text dataset and store it in a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # The first 100 characters we saw earlier, now as integers

## 2. Splitting Data into Training and Validation Sets

We'll use the first 90% of the data for training and the remaining 10% for validation.

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(f"Training data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")

## Testing the Model Components

In [3]:
import torch
from torch import Tensor

torch.manual_seed(0) # type: ignore

In [4]:
import importlib
import model as model_module
importlib.reload(model_module)

Head = model_module.Head
block_size = model_module.block_size
n_embd = model_module.n_embd

print("block_size:", block_size, "n_embd:", n_embd)

block_size: 8 n_embd: 32


### Head

In [ ]:
# Basic forward pass + shape checks
B, T, C = 2, min(5, block_size), n_embd
head_size = 16

head = Head(head_size=head_size)
x = torch.randn(B, T, C)

out: Tensor = head(x)

print("x.shape:", x.shape)
print("out.shape:", out.shape)
assert out.shape == (B, T, head_size)

In [ ]:
# Causality check
# Changing the LAST token should NOT affect outputs at earlier timesteps.
with torch.no_grad():
    x2 = x.clone()
    x2[:, -1, :] += 999.0  # big perturbation at the final position

    out1: Tensor = head(x)
    out2: Tensor = head(x2)

    max_diff_earlier = (out1[:, :-1, :] - out2[:, :-1, :]).abs().max().item()
    max_diff_last = (out1[:, -1, :] - out2[:, -1, :]).abs().max().item()

print("max_diff earlier positions:", max_diff_earlier)
print("max_diff last position:", max_diff_last)

assert max_diff_earlier < 1e-6, "Causal mask appears broken: earlier outputs changed."
assert max_diff_last > 1e-6, "Unexpected: last position did not change after perturbation."

In [ ]:
# Gradient flow check
head = Head(head_size=head_size)
x = torch.randn(B, T, C, requires_grad=True)

out: Tensor = head(x)
loss: Tensor = out.pow(2).mean()
loss.backward() # type: ignore

assert x.grad is not None
assert head.key.weight.grad is not None

print("x.grad max abs:", x.grad.abs().max().item())
print("key.weight grad max abs:", head.key.weight.grad.abs().max().item())

assert torch.isfinite(x.grad).all()
assert torch.isfinite(head.key.weight.grad).all()

### MultiHeadAttention

In [5]:
importlib.reload(model_module)
MultiHeadAttention = model_module.MultiHeadAttention

# --- Hyperparameters for the test ---
num_heads = 4
B, T, C = 2, min(5, block_size), n_embd # (Batch, Time, Channels)

# --- Test Forward Pass ---
mha = MultiHeadAttention(num_heads=num_heads)
x = torch.randn(B, T, C)
out: Tensor = mha(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)
assert out.shape == (B, T, C), "Output shape is incorrect!"
print("Forward pass and shape check successful.")

Input shape: torch.Size([2, 5, 32])
Output shape: torch.Size([2, 5, 32])
Forward pass and shape check successful.


In [6]:
# --- Test Gradient Flow ---
mha = MultiHeadAttention(num_heads=num_heads)
x = torch.randn(B, T, C, requires_grad=True)

out: Tensor = mha(x)
loss: Tensor = out.pow(2).mean()
loss.backward()  # type: ignore

head1: Head = mha.heads[0] # type: ignore

assert x.grad is not None
assert mha.proj.weight.grad is not None
assert head1.key.weight.grad is not None

print("x.grad max abs:", x.grad.abs().max().item())
print("proj.weight grad max abs:", mha.proj.weight.grad.abs().max().item())
print(
    "head[0].key.weight grad max abs:", head1.key.weight.grad.abs().max().item()
)

assert torch.isfinite(x.grad).all()
assert torch.isfinite(mha.proj.weight.grad).all()
assert torch.isfinite(head1.key.weight.grad).all()
print("Gradient flow check successful.")

x.grad max abs: 0.003983018454164267
proj.weight grad max abs: 0.020697908475995064
head[0].key.weight grad max abs: 0.0016259672120213509
Gradient flow check successful.
